In [1]:
import os, sqlite3
import numpy as np
import pandas as pd
from pathlib import Path

db_path = Path(r"C:\Users\DIVINE-GIFT\Documents\data-science-lab\data-acquisition\chinook-database\ChinookDatabase\DataSources\Chinook_Sqlite.sqlite")

assert db_path.exists(), f"File not found: {db_path}"
print("Size (KB):", round(db_path.stat().st_size / 1024))   # expect 984

con = sqlite3.connect(db_path)

Size (KB): 984


In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

db_path = Path(r"C:\Users\DIVINE-GIFT\Documents\data-science-lab\data-acquisition\chinook-database\ChinookDatabase\DataSources\Chinook_Sqlite.sqlite")
assert db_path.exists(), f"File not found: {db_path}"

con = sqlite3.connect(db_path)

# Discover every table, then read each one
table_names = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;", con
)["name"].tolist()

db = {t: pd.read_sql_query(f'SELECT * FROM "{t}";', con) for t in table_names}

# Summary of what was loaded
summary = pd.DataFrame({
    "rows": {t: df.shape[0] for t, df in db.items()},
    "cols": {t: df.shape[1] for t, df in db.items()},

In [1]:
})
display(summary)

,rows,cols
Album,347,3
Artist,275,2
Customer,59,13
Employee,8,15
Genre,25,2
Invoice,412,9
InvoiceLine,2240,5
MediaType,5,2
Playlist,18,2
PlaylistTrack,8715,2


In [2]:
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;", con
)
display(tables)                      # expect 11 tables

customers = pd.read_sql_query("SELECT * FROM Customer;", con)
invoices  = pd.read_sql_query("SELECT * FROM Invoice;", con)
lines     = pd.read_sql_query("SELECT * FROM InvoiceLine;", con)

print(customers.shape, invoices.shape, lines.shape)   # about (59, 13) (412, 9) (2240, 5)

,name
0,Album
1,Artist
2,Customer
3,Employee
4,Genre
5,Invoice
6,InvoiceLine
7,MediaType
8,Playlist
9,PlaylistTrack


(59, 13) (412, 9) (2240, 5)


In [3]:
q = '''
SELECT InvoiceId, CustomerId, InvoiceDate, BillingCountry, Total
FROM Invoice
WHERE Total > 5
ORDER BY Total DESC
'''
large_invoices = pd.read_sql_query(q, con, parse_dates=["InvoiceDate"])
large_invoices.info()
large_invoices.head()

<class 'pandas.DataFrame'>
RangeIndex: 179 entries, 0 to 178
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   InvoiceId       179 non-null    int64         
 1   CustomerId      179 non-null    int64         
 2   InvoiceDate     179 non-null    datetime64[us]
 3   BillingCountry  179 non-null    str           
 4   Total           179 non-null    float64       
dtypes: datetime64[us](1), float64(1), int64(2), str(1)
memory usage: 8.2 KB


,InvoiceId,CustomerId,InvoiceDate,BillingCountry,Total
0,404,6,2025-11-13,Czech Republic,25.86
1,299,26,2024-08-05,USA,23.86
2,96,45,2022-02-18,Hungary,21.86
3,194,46,2023-04-28,Ireland,21.86
4,89,7,2022-01-18,Austria,18.86


In [4]:
print("Before:", invoices.shape, "After:", large_invoices.shape)
print("Missing:\n", large_invoices.isna().sum())
print("Duplicate rows:", large_invoices.duplicated().sum())

assert large_invoices["InvoiceId"].is_unique
assert large_invoices["Total"].min() > 5
assert len(large_invoices) == (invoices["Total"] > 5).sum()   # SQL filter matches Pandas filter

Before: (412, 9) After: (179, 5)
Missing:
 InvoiceId         0
CustomerId        0
InvoiceDate       0
BillingCountry    0
Total             0
dtype: int64
Duplicate rows: 0


In [5]:
large_invoices[["InvoiceId", "Total"]].head()


,InvoiceId,Total
0,404,25.86
1,299,23.86
2,96,21.86
3,194,21.86
4,89,18.86


In [6]:
usa_can = large_invoices.query("BillingCountry in ['USA', 'Canada']")
usa_can["BillingCountry"].value_counts()

BillingCountry
USA       40
Canada    24
Name: count, dtype: int64

In [7]:
for name, frame, key in [
    ("Customer", customers, "CustomerId"),
    ("Invoice", invoices, "InvoiceId"),
    ("InvoiceLine", lines, "InvoiceLineId"),
]:
    print(name, frame.shape, "key unique:", frame[key].is_unique,
          "duplicates:", frame.duplicated().sum())
    print(frame.isna().sum().sort_values(ascending=False).head())
    print()

Customer (59, 13) key unique: True duplicates: 0
Company       49
Fax           47
State         29
PostalCode     4
Phone          1
dtype: int64

Invoice (412, 9) key unique: True duplicates: 0
BillingState         202
BillingPostalCode     28
InvoiceId              0
CustomerId             0
InvoiceDate            0
dtype: int64

InvoiceLine (2240, 5) key unique: True duplicates: 0
InvoiceLineId    0
InvoiceId        0
TrackId          0
UnitPrice        0
Quantity         0
dtype: int64



In [8]:
customers.info()
print(invoices.dtypes)
print(lines.describe())
print(customers.nunique())

<class 'pandas.DataFrame'>
RangeIndex: 59 entries, 0 to 58
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   CustomerId    59 non-null     int64
 1   FirstName     59 non-null     str  
 2   LastName      59 non-null     str  
 3   Company       10 non-null     str  
 4   Address       59 non-null     str  
 5   City          59 non-null     str  
 6   State         30 non-null     str  
 7   Country       59 non-null     str  
 8   PostalCode    55 non-null     str  
 9   Phone         58 non-null     str  
 10  Fax           12 non-null     str  
 11  Email         59 non-null     str  
 12  SupportRepId  59 non-null     int64
dtypes: int64(2), str(11)
memory usage: 11.7 KB
InvoiceId              int64
CustomerId             int64
InvoiceDate              str
BillingAddress           str
BillingCity              str
BillingState             str
BillingCountry           str
BillingPostalCode        str
Total        

In [9]:
inv = invoices.copy()
inv["InvoiceDate"]   = pd.to_datetime(inv["InvoiceDate"], errors="coerce")
inv["invoice_year"]  = inv["InvoiceDate"].dt.year
inv["invoice_month"] = inv["InvoiceDate"].dt.to_period("M").astype(str)
inv["Total"]         = pd.to_numeric(inv["Total"], errors="coerce")
assert inv["InvoiceId"].is_unique

In [10]:
print(invoices.shape, "->", inv.shape)
print("Bad dates:", inv["InvoiceDate"].isna().sum(), "| Bad totals:", inv["Total"].isna().sum())

assert len(inv) == len(invoices)
assert np.isclose(inv["Total"].sum(), invoices["Total"].sum())
inv[["InvoiceId", "InvoiceDate", "invoice_year", "invoice_month"]].head()

(412, 9) -> (412, 11)
Bad dates: 0 | Bad totals: 0


,InvoiceId,InvoiceDate,invoice_year,invoice_month
0,1,2021-01-01,2021,2021-01
1,2,2021-01-02,2021,2021-01
2,3,2021-01-03,2021,2021-01
3,4,2021-01-06,2021,2021-01
4,5,2021-01-11,2021,2021-01


In [11]:
inv.pivot_table(index="invoice_year", columns="BillingCountry",
                values="Total", aggfunc="sum").head()

inv["invoice_size"] = pd.cut(inv["Total"], [0, 2, 6, 10, 100],
                             labels=["small", "medium", "large", "xl"])
inv["invoice_size"].value_counts()

invoice_size
small     170
medium    119
xl         64
large      59
Name: count, dtype: int64

In [12]:
country = (
    inv.groupby("BillingCountry", dropna=False)
       .agg(invoices=("InvoiceId", "nunique"),
            customers=("CustomerId", "nunique"),
            revenue=("Total", "sum"),
            avg_invoice=("Total", "mean"))
       .sort_values("revenue", ascending=False)
       .reset_index()
)

customer = (
    inv.groupby("CustomerId")
       .agg(invoices=("InvoiceId", "nunique"), revenue=("Total", "sum"))
       .sort_values("revenue", ascending=False)
)

display(country.head(10))
display(customer.head(10))

,BillingCountry,invoices,customers,revenue,avg_invoice
0,USA,91,13,523.06,5.747912
1,Canada,56,8,303.96,5.427857
2,France,35,5,195.10,5.574286
3,Brazil,35,5,190.10,5.431429
4,Germany,28,4,156.48,5.588571
5,United Kingdom,21,3,112.86,5.374286
6,Czech Republic,14,2,90.24,6.445714
7,Portugal,14,2,77.24,5.517143
8,India,13,2,75.26,5.789231
9,Chile,7,1,46.62,6.660000


,invoices,revenue
CustomerId,,
6,7,49.62
26,7,47.62
57,7,46.62
45,7,45.62
46,7,45.62
28,7,43.62
37,7,43.62
24,7,43.62
7,7,42.62


In [13]:
assert np.isclose(country["revenue"].sum(), inv["Total"].sum())
assert np.isclose(customer["revenue"].sum(), inv["Total"].sum())
assert country["invoices"].sum() == len(inv)
print(country.shape, customer.shape)

(24, 5) (59, 2)


In [14]:
inv["country_share"] = inv["Total"] / inv.groupby("BillingCountry")["Total"].transform("sum")
inv[["InvoiceId", "BillingCountry", "Total", "country_share"]].head()

,InvoiceId,BillingCountry,Total,country_share
0,1,Germany,1.98,0.012653
1,2,Norway,3.96,0.099950
2,3,Belgium,5.94,0.157895
3,4,Canada,8.91,0.029313
4,5,USA,13.86,0.026498


In [15]:
tracks = pd.read_sql_query("SELECT TrackId, Name, AlbumId, GenreId, UnitPrice FROM Track;", con)
genres = pd.read_sql_query("SELECT GenreId, Name AS GenreName FROM Genre;", con)
assert tracks["TrackId"].is_unique
assert genres["GenreId"].is_unique

x = (lines
     .merge(tracks, on="TrackId", how="left", validate="many_to_one",
            suffixes=("_sale", "_catalog"), indicator="track_merge")
     .merge(genres, on="GenreId", how="left", validate="many_to_one"))

x["line_revenue"] = x["UnitPrice_sale"] * x["Quantity"]

genre_sales = (x.groupby("GenreName", dropna=False)
                .agg(lines=("InvoiceLineId", "size"),
                     units=("Quantity", "sum"),
                     revenue=("line_revenue", "sum"))
                .sort_values("revenue", ascending=False))
genre_sales.head(10)

,lines,units,revenue
GenreName,,,
Rock,835,835,826.65
Latin,386,386,382.14
Metal,264,264,261.36
Alternative & Punk,244,244,241.56
TV Shows,47,47,93.53
Jazz,80,80,79.20
Blues,61,61,60.39
Drama,29,29,57.71
Classical,41,41,40.59


In [16]:
print(lines.shape, "->", x.shape)
assert len(x) == len(lines)                       # no rows duplicated or lost
assert (x["track_merge"] == "both").all()         # every line matched a track
assert x["GenreName"].notna().all()               # every track has a genre
assert np.isclose(x["line_revenue"].sum(), invoices["Total"].sum())

(2240, 5) -> (2240, 12)
